In [28]:
import pretty_midi
import os
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
import datetime
from torch.utils.data import random_split
import glob
import random
import torch.nn.functional as F

In [29]:
def generate_music(model, tokenizer, seed_seq, gen_length=200, temperature=1.0, device='cuda'):
    """
    model: Your trained SotaMusicTransformer
    tokenizer: The MaestroTokenizer used during training
    seed_seq: A starting sequence of tokens shape (64, 4)
    gen_length: How many new notes to generate
    temperature: Higher = more "creative/chaotic", Lower = more "conservative"
    """
    model.eval()
    # Initial sequence from seed
    generated_tokens = list(seed_seq) 
    
    with torch.no_grad():
        for _ in range(gen_length):
            # Take the last WINDOW_SIZE notes
            input_window = np.array(generated_tokens[-64:])
            
            # Prepare tensors
            p = torch.LongTensor(input_window[:, 0]).unsqueeze(0).to(device)
            s = torch.LongTensor(input_window[:, 1]).unsqueeze(0).to(device)
            d = torch.LongTensor(input_window[:, 2]).unsqueeze(0).to(device)
            v = torch.LongTensor(input_window[:, 3]).unsqueeze(0).to(device)
            
            # Forward pass
            out_p, out_s, out_d, out_v = model(p, s, d, v)
            
            # Apply Temperature and Sample (instead of just Argmax)
            def sample(logits, temp):
                probs = F.softmax(logits / temp, dim=-1)
                return torch.multinomial(probs, 1).item()

            # Sample each feature
            next_note = [
                sample(out_p, temperature),
                sample(out_s, temperature),
                sample(out_d, temperature),
                sample(out_v, temperature)
            ]
            
            generated_tokens.append(next_note)
            
    return np.array(generated_tokens)

In [30]:
class MusicSotaTransformer(nn.Module):
    def __init__(self, d_model=256, nhead=8, num_layers=6, max_seq_len=128):
        super().__init__()
        
        # 1. Vocabulary
        self.pitch_embed = nn.Embedding(128, d_model)   # 128 MIDI Notes
        self.step_embed = nn.Embedding(100, d_model)    # 100 Time Bins
        self.dur_embed = nn.Embedding(100, d_model)     # 100 Duration-Bins
        self.vel_embed = nn.Embedding(32, d_model)      # 32 Velocity-Bins
        
        # 2. Merge into one Layer
        self.feature_merger = nn.Linear(d_model * 4, d_model)
        
        # 3. Position Encoding: Significance for note at the start and at teh end
        self.pos_embed = nn.Parameter(torch.zeros(1, max_seq_len, d_model))
        
        # 4. Transformer Core (GPT-style Decoder)
        #
        decoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=1024, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(decoder_layer, num_layers=num_layers)
        
        # 5. seperate outputs for each value (Multi-Task Learning)
        #
        self.head_pitch = nn.Linear(d_model, 128)
        self.head_step = nn.Linear(d_model, 100)
        self.head_dur = nn.Linear(d_model, 100)
        self.head_vel = nn.Linear(d_model, 32)

    def forward(self, p, s, d, v):
        # get embeddings
        p_e = self.pitch_embed(p)
        s_e = self.step_embed(s)
        d_e = self.dur_embed(d)
        v_e = self.vel_embed(v)
        
        # combine features
        x = torch.cat((p_e, s_e, d_e, v_e), dim=-1)
        
        # 1. Get current sequence length (e.g., 64)
        seq_len = x.size(1)
        
        # 2. FIX: Slice the 128-length pos_embed to match the 64-length x
        x = self.feature_merger(x) + self.pos_embed[:, :seq_len, :]
        
        # Transformer
        x = self.transformer(x)
        
        # use last token for prediction
        last_token = x[:, -1, :]
        
        # prediction for every output head
        return (
            self.head_pitch(last_token),
            self.head_step(last_token),
            self.head_dur(last_token),
            self.head_vel(last_token)
        )

In [31]:
def tokens_to_midi(tokens, tokenizer, output_file="output.mid"):
    pm = pretty_midi.PrettyMIDI()
    piano = pretty_midi.Instrument(program=0) # Acoustic Grand Piano
    
    current_time = 0
    
    for tok in tokens:
        p_idx, s_idx, d_idx, v_idx = tok
        
        # --- DETOKENIZATION LOGIC ---
        
        # 1. Pitch: Direct mapping
        pitch = int(p_idx)
        
        # 2. Step & Duration: Map indices back to the time_bins values
        # We use the index to grab the corresponding value from your logspace array
        step = tokenizer.time_bins[s_idx]
        duration = tokenizer.time_bins[d_idx]

        duration = duration * 2
        step = step * 2
        
        # 3. Velocity: Reverse the // 4 quantization
        # We multiply by 4 (and add 2 to hit the middle of the quantized range)
        velocity = int(v_idx * 4 + 2)
        
        # --- MIDI CONSTRUCTION ---
        
        # Calculate start and end times
        start_time = current_time + step
        end_time = start_time + duration
        
        # Safety check: Ensure start < end
        if end_time <= start_time:
            end_time = start_time + 0.1
            
        note = pretty_midi.Note(
            velocity=np.clip(velocity, 0, 127),
            pitch=np.clip(pitch, 0, 127),
            start=float(start_time),
            end=float(end_time)
        )
        piano.notes.append(note)
        
        # Update pointer (the "step" is relative to the previous note's start)
        current_time = start_time
        
    pm.instruments.append(piano)
    pm.write(output_file)
    print(f"Successfully saved MIDI to {output_file}")

In [32]:
class MaestroTokenizer:
    def __init__(self):
        # 100 log-spaced bins for time-related features (0.001s to 4s)
        self.time_bins = np.logspace(np.log10(0.001), np.log10(4.0), num=100)

    def tokenize_file(self, file_path):
        try:
            pm = pretty_midi.PrettyMIDI(file_path)
            # MAESTRO is usually all piano, but we merge just in case
            notes = []
            for instrument in pm.instruments:
                if not instrument.is_drum:
                    notes.extend(instrument.notes)
            
            # Sort notes by start time
            notes.sort(key=lambda x: x.start)
            
            tokenized_events = []
            for i in range(len(notes)):
                n = notes[i]
                
                # Feature 1: Pitch (0-127)
                p_idx = int(np.clip(n.pitch, 0, 127))
                
                # Feature 2: Step (Time since previous note started)
                step = n.start - notes[i-1].start if i > 0 else 0
                s_idx = np.clip(np.digitize(step, self.time_bins) - 1, 0, 99)
                
                # Feature 3: Duration
                dur = n.end - n.start
                d_idx = np.clip(np.digitize(dur, self.time_bins) - 1, 0, 99)
                
                # Feature 4: Velocity (Quantized 128 -> 32)
                v_idx = int(np.clip(n.velocity // 4, 0, 31))
                
                tokenized_events.append([p_idx, s_idx, d_idx, v_idx])
                
            return np.array(tokenized_events, dtype=np.int64)
        except Exception as e:
            # print(f"Error processing {file_path}: {e}")
            return None

In [33]:
def load_model(checkpoint_path, model_class, device, **model_kwargs):
    """
    Loads the model weights and handles DataParallel unwrapping.
    """
    # 1. Initialize the architecture
    model = model_class(**model_kwargs).to(device)
    
    # 2. Load the file
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    # 3. Handle DataParallel (remove 'module.' prefix if it exists)
    state_dict = checkpoint['model_state_dict']
    from collections import OrderedDict
    new_state_dict = OrderedDict()
    
    for k, v in state_dict.items():
        name = k[7:] if k.startswith('module.') else k  # remove 'module.'
        new_state_dict[name] = v
        
    # 4. Load weights into the model
    model.load_state_dict(new_state_dict)
    model.eval() # Set to evaluation mode
    
    print(f"Model loaded from {checkpoint_path} (Epoch: {checkpoint.get('epoch', 'unknown')}, Loss: {checkpoint.get('loss', 'unknown'):.4f})")
    return model

In [36]:
# --- CONFIGURATION ---
MAESTRO_PATH = "/home/nour_mbb/assignment3/datasets/midi/maestro/maestro-v3.0.0"
WINDOW_SIZE = 64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Initialize Tokenizer
tokenizer = MaestroTokenizer()

# 2. Pick exactly ONE random file from the directory
# This looks for all .midi or .mid files recursively
all_midi_files = glob.glob(os.path.join(MAESTRO_PATH, "**/*.mid*"), recursive=True)

if not all_midi_files:
    raise FileNotFoundError(f"No MIDI files found in {MAESTRO_PATH}")

random_file = random.choice(all_midi_files)
print(f"Selected seed file: {os.path.basename(random_file)}")

# 3. Tokenize only this specific file
# This is very fast and uses almost no RAM
raw_tokens = tokenizer.tokenize_file(random_file)

if raw_tokens is None or len(raw_tokens) < WINDOW_SIZE + 1:
    raise ValueError("The selected file is too short to be used as a seed.")

# 4. Extract a random seed window from this file
# We pick a random starting point within the file
start_idx = random.randint(0, len(raw_tokens) - WINDOW_SIZE - 1)
seed_np = raw_tokens[start_idx : start_idx + WINDOW_SIZE] # Shape (64, 4)

print(f"Successfully extracted seed of shape {seed_np.shape}")

# --- NOW YOU CAN LOAD YOUR MODEL ---
model = load_model(
    checkpoint_path="checkpoint_epoch_20.pth",
    model_class=MusicSotaTransformer,
    device=DEVICE,
    d_model=256, 
    nhead=8, 
    num_layers=6, 
    max_seq_len=128
)

# --- AND GENERATE ---
gen_tokens = generate_music(model, tokenizer, seed_np, gen_length=400, temperature=1.2, device=DEVICE)
tokens_to_midi(gen_tokens, tokenizer, "random_seed_output.mid")

Selected seed file: MIDI-UNPROCESSED_16-18_R1_2014_MID--AUDIO_17_R1_2014_wav--1.midi
Successfully extracted seed of shape (64, 4)
Model loaded from checkpoint_epoch_20.pth (Epoch: 19, Loss: 14.9523)


/tmp/ipykernel_2270611/3222361573.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location=device)


Successfully saved MIDI to random_seed_output.mid
